In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader

In [34]:
# Example usage
embed_dim = 16  # Embedding dimension
num_heads = 4  # Number of attention heads
num_layers = 2  # Number of transformer layers
dropout = 0.1  # Dropout rate

graph_input_dim = 4  # Number of rows in the graph
text_vocab_size = 26  # Vocabulary size for text
graph_data = torch.randn(32,10, graph_input_dim)
text_input = torch.randint(0, text_vocab_size, (32, 5))

### Model

In [27]:
class GraphToTextTransformer(nn.Module):
    def __init__(self, graph_input_dim, text_vocab_size, embed_dim, num_heads, num_layers, dropout=0.1):
        super(GraphToTextTransformer, self).__init__()
        self.embed_dim = embed_dim

        # Encoder: Linear layer to embed graph columns
        self.encoder_embedding = nn.Linear(graph_input_dim, embed_dim)
    
        # Decoder: Embedding layer for text input
        self.decoder_embedding = nn.Embedding(text_vocab_size, embed_dim)


        # Transformer model
        self.transformer = nn.Transformer(
            d_model=embed_dim,
            nhead=num_heads,
            num_encoder_layers=num_layers,
            num_decoder_layers=num_layers,
            dropout=dropout
        )

        # Output layer to map decoder output to text vocabulary
        self.output_layer = nn.Linear(embed_dim, text_vocab_size)

    def forward(self, graph_data, text_input, src_mask=None, tgt_mask=None):
        """
        Args:
            graph_data: Tensor of shape (batch_size, seq_len, graph_input_dim)
            text_input: Tensor of shape (batch_size, tgt_seq_len)
            src_mask: Optional mask for the encoder input
            tgt_mask: Optional mask for the decoder input

        Returns:
            Tensor of shape (batch_size, tgt_seq_len, text_vocab_size)
        """
        # Encode graph data
        graph_encoded = self.encoder_embedding(graph_data)  # (batch_size, seq_len, embed_dim)
        graph_encoded = graph_encoded.permute(1, 0, 2)  # (seq_len, batch_size, embed_dim)

        # Embed text input
        text_embedded = self.decoder_embedding(text_input)  # (batch_size, tgt_seq_len, embed_dim)
        text_embedded = text_embedded.permute(1, 0, 2)  # (tgt_seq_len, batch_size, embed_dim)

        # Pass through transformer
        transformer_output = self.transformer(
            src=graph_encoded,
            tgt=text_embedded,
            src_mask=src_mask,
            tgt_mask=tgt_mask
        )  # (tgt_seq_len, batch_size, embed_dim)

        # Map to text vocabulary
        output = self.output_layer(transformer_output)  # (tgt_seq_len, batch_size, text_vocab_size)
        return output.permute(1, 0, 2)  # (batch_size, tgt_seq_len, text_vocab_size)

In [28]:
# Load the model checkpoint
def load_checkpoint(model, optimizer, filename):
    checkpoint = torch.load(filename)
    model.load_state_dict(checkpoint['state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer'])

In [29]:
# Initialize model
model = GraphToTextTransformer(
    graph_input_dim, 
    text_vocab_size, 
    embed_dim, 
    num_heads, 
    num_layers, 
    dropout)

/home/nithira/gen_ai/.venv/lib/python3.10/site-packages/torch/nn/modules/transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


In [30]:
learning_rate = 0.001
num_epochs = 10000

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [35]:
# Training loop
for epoch in range(num_epochs):
    model.train()  # Set the model to training mode

    # Forward pass
    output = model(graph_data, text_input[:, :-1])  # Exclude the last token for input
    output = output.reshape(-1, text_vocab_size)  # Reshape for loss calculation
    target = text_input[:, 1:].reshape(-1)  # Exclude the first token for target

    # Compute loss
    loss = criterion(output, target)

    # Backward pass and optimization
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Print loss for the epoch
    if (epoch + 1) % 100 == 0:  # Print every 100 epochs
        print(f"Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item():.4f}")

Epoch [100/10000], Loss: 0.1767
Epoch [200/10000], Loss: 0.1234


KeyboardInterrupt: 